# SQL Query Optimization
---

<div style="padding:15px;border-left:8px solid #f093fb;background:#fdf0ff;border-radius:4px;">
<strong>Core Insight:</strong> Query optimization separates analysts from engineers.
Know EXPLAIN output, why indexes matter, and how the planner decides between
nested loop, hash join, and merge join. A single missing index can turn a 0.3-second
query into a 45-second table scan.
</div>

### Why It Matters
Slow queries in production cause SLA breaches. At Citi, a poorly indexed telemetry
query scanned 500M rows — taking 45 seconds. After adding a composite index and
enabling partition pruning: **0.3 seconds**.

## 🧠 Three Mental Models for Query Optimization

| Model | The Insight |
|-------|------------|
| **The Librarian** | An index is like a book's index — you jump to the page directly instead of reading every page |
| **The Join Planner** | The query planner estimates row counts and chooses: Nested Loop (small tables), Hash Join (large unsorted), Merge Join (pre-sorted) |
| **The Partition Pruner** | Partitioning by date means a query for "last 7 days" only reads 7 of 365 partitions — 98% skip |

**The Golden Rule:** Optimize for cardinality. High-cardinality columns (user_id, timestamp) make
excellent index leading columns. Low-cardinality (status, boolean) do not.

In [ ]:
-- EXPLAIN: Read the query plan before optimizing anything
-- PostgreSQL syntax (similar in most databases)

-- Step 1: Get the raw plan (no execution)
EXPLAIN
SELECT s.server_name, m.metric_value, m.collected_at
FROM fact_monitoring m
JOIN dim_server s ON m.server_id = s.server_id
WHERE m.collected_at >= '2024-01-01'
  AND m.metric_type = 'cpu_pct'
ORDER BY m.collected_at DESC;

-- Step 2: Actually run it and get real timings
EXPLAIN ANALYZE
SELECT s.server_name, m.metric_value, m.collected_at
FROM fact_monitoring m
JOIN dim_server s ON m.server_id = s.server_id
WHERE m.collected_at >= '2024-01-01'
  AND m.metric_type = 'cpu_pct'
ORDER BY m.collected_at DESC;

-- What to look for in the output:
-- "Seq Scan" = reading every row (BAD for large tables)
-- "Index Scan" = using an index (GOOD)
-- "Hash Join" = building a hash table of smaller side (GOOD for large joins)
-- "rows=X" vs "actual rows=Y" -- big mismatch = stale statistics (run ANALYZE)

## 📊 Index Types and When to Use Them

| Index Type | When to Use | Example |
|-----------|------------|---------|
| **B-Tree (default)** | Range queries, equality, ORDER BY | `WHERE collected_at > '2024-01-01'` |
| **Hash** | Equality only, no range | `WHERE server_id = 42` (exact match) |
| **Composite** | Multi-column WHERE / ORDER BY | `WHERE env = 'prod' AND collected_at > ...` |
| **Covering** | Query only needs indexed columns | Avoid table heap access entirely |
| **Partial** | Filter on a subset of rows | `WHERE status = 'active'` (only index active rows) |

### Composite Index Column Order Rule
Put the **most selective** (highest cardinality) column FIRST.
The leading column is used for range scans; trailing columns narrow within matches.

```sql
-- Good: collected_at is high-cardinality (many distinct timestamps)
CREATE INDEX idx_monitoring_time_env ON fact_monitoring(collected_at, env);

-- Bad: env has 3 values (dev/staging/prod) — low cardinality leading column
CREATE INDEX idx_monitoring_env_time ON fact_monitoring(env, collected_at);
```

In [ ]:
-- Index creation examples

-- Basic index on timestamp column
CREATE INDEX idx_monitoring_collected_at
ON fact_monitoring(collected_at);

-- Composite index: most selective column first
CREATE INDEX idx_monitoring_time_type
ON fact_monitoring(collected_at DESC, metric_type);

-- Covering index: include extra columns to avoid heap access
-- Query only needs these columns — index covers the full query
CREATE INDEX idx_monitoring_covering
ON fact_monitoring(server_id, collected_at DESC)
INCLUDE (metric_value, metric_type);

-- Partial index: only index active servers (assume 80% are decommissioned)
CREATE INDEX idx_server_active
ON dim_server(server_name, env)
WHERE is_active = TRUE;

-- BEFORE: Seq Scan on 500M rows = 45 seconds
-- AFTER:  Index Scan on 2M matching rows = 0.3 seconds

-- Check index usage
SELECT schemaname, tablename, indexname, idx_scan, idx_tup_read
FROM pg_stat_user_indexes
WHERE tablename = 'fact_monitoring'
ORDER BY idx_scan DESC;

## 🔗 Join Strategies: How the Planner Decides

### Nested Loop Join
- Best when: inner table is **small** or has an **index** on the join key
- Cost: O(outer_rows × inner_rows) — catastrophic for large tables

### Hash Join
- Best when: one table fits in memory, no useful index
- Cost: O(n + m) — build hash table of smaller side, probe with larger side
- Forces a full scan of both tables

### Merge Join
- Best when: **both sides are pre-sorted** on the join key
- Cost: O(n log n + m log m) — efficient when data already sorted

```sql
-- Force a specific join type for testing (PostgreSQL):
SET enable_hashjoin = OFF;   -- force merge or nested loop
SET enable_mergejoin = OFF;  -- force hash or nested loop

-- Check what plan was chosen:
EXPLAIN ANALYZE
SELECT * FROM fact_monitoring m
JOIN dim_server s ON m.server_id = s.server_id;
```

In [ ]:
-- Query optimization in practice: before and after

-- ══════════════════════════════════════════════
-- SCENARIO: Monthly capacity report for active servers
-- Table: fact_monitoring (500M rows), partitioned by month
-- ══════════════════════════════════════════════

-- BEFORE: 45 seconds, Seq Scan
SELECT
    s.server_name,
    s.environment,
    AVG(m.metric_value) AS avg_cpu,
    MAX(m.metric_value) AS peak_cpu
FROM fact_monitoring m
JOIN dim_server s ON m.server_id = s.server_id
WHERE m.metric_type = 'cpu_pct'
  AND m.collected_at BETWEEN '2024-01-01' AND '2024-01-31'
  AND s.is_active = TRUE
GROUP BY s.server_name, s.environment
HAVING AVG(m.metric_value) > 70
ORDER BY avg_cpu DESC;

-- Optimization steps applied:
-- 1. Add composite index: (collected_at, metric_type, server_id)
-- 2. Add partial index on dim_server where is_active = TRUE
-- 3. Partition fact_monitoring by month → January partition only
-- 4. Add covering index to include metric_value

-- AFTER: 0.3 seconds, Index Scan + Partition Pruning
-- Same query, same result — different execution plan

-- Verify partition pruning happened:
EXPLAIN SELECT * FROM fact_monitoring
WHERE collected_at BETWEEN '2024-01-01' AND '2024-01-31';
-- Look for: "Partitions: fact_monitoring_2024_01"
-- NOT: scanning all partitions

## 🎤 5 Interview Q&A

**Q1: What does EXPLAIN ANALYZE tell you that EXPLAIN alone doesn't?**
EXPLAIN shows the *estimated* plan — what the planner thinks will happen.
EXPLAIN ANALYZE actually *executes* the query and shows real row counts and timings.
The gap between estimated and actual rows reveals stale statistics — fix with `ANALYZE table_name`.

---

**Q2: When would you NOT add an index?**
(1) Tables with < 10K rows — sequential scan is faster than index overhead.
(2) Columns with low cardinality (boolean, 3-value status) — the index doesn't narrow the search enough.
(3) Write-heavy tables — every INSERT/UPDATE/DELETE must update all indexes, slowing writes.
(4) Columns never used in WHERE/JOIN/ORDER BY clauses.

---

**Q3: What is a covering index and why does it matter?**
A covering index includes all columns the query needs — so the planner never touches the heap (main table).
Example: `CREATE INDEX idx ON orders(customer_id) INCLUDE (total, created_at)`.
A query selecting only those columns hits the index only — typically 2-5x faster than an index + heap lookup.

---

**Q4: What is partition pruning and when does it help?**
Partition pruning means the query planner skips partitions whose data cannot match the WHERE clause.
If `fact_monitoring` is partitioned by month and you query `WHERE collected_at >= '2024-01-01'`,
the planner reads only January's partition — not all 12. Most effective when queries have tight
date range filters, which is common in time-series data.

---

**Q5: What is cardinality and why does it affect index design?**
Cardinality = the number of distinct values in a column.
High cardinality (user_id: millions of distinct values) → index is very selective → each lookup eliminates most rows.
Low cardinality (status: 'active'/'inactive') → index is not selective → the planner may prefer a full scan.
Rule: put high-cardinality columns first in composite indexes.

## 📚 Key Terms

| Term | Definition |
|------|------------|
| **Sequential Scan (Seq Scan)** | Reading every row in the table from disk — O(n), appropriate only for small tables or when most rows match |
| **Index Scan** | Using a B-tree index to jump to matching rows — O(log n + k) where k is result rows |
| **Hash Join** | Build a hash table of the smaller relation, probe with the larger — O(n+m) |
| **Nested Loop Join** | For each row in the outer table, scan the inner — O(n×m), good only with indexed inner |
| **Merge Join** | Sort both sides, merge — O(n log n + m log m), ideal when data already sorted |
| **Composite Index** | Index on multiple columns — column order matters: most selective first |
| **Covering Index** | Index that includes all columns the query needs — avoids heap access |
| **Partial Index** | Index on a filtered subset of rows — `CREATE INDEX ... WHERE status = 'active'` |
| **Partition Pruning** | Query planner skips partitions that can't match the WHERE clause |
| **Cardinality** | Number of distinct values in a column — high cardinality → selective index |
| **EXPLAIN ANALYZE** | Run the query and show actual execution stats (rows, timing) vs estimated |
| **Statistics** | Row count / distribution estimates the planner uses — updated by `ANALYZE` |

## 💼 The Citi Narrative

**Context:** Citi APM telemetry database — `fact_monitoring` table with 500M rows,
storing CPU/memory/response time for 6,000 servers at 1-minute intervals.

**The Problem:** Monthly capacity report took 45 seconds. Analysts ran it during business hours.
Every run locked resources and frustrated the team.

**Root Cause (from EXPLAIN ANALYZE):**
- Sequential scan on `fact_monitoring` (500M rows) — no index on `collected_at`
- Hash Join with `dim_server` — 6,000 rows, small enough but still building a hash table
- Estimates wildly off: planner estimated 50K rows, actual was 2M rows → stale statistics

**The Fix — Three Changes:**
1. `CREATE INDEX idx_monitoring_time_type ON fact_monitoring(collected_at DESC, metric_type)` — composite index
2. `ANALYZE fact_monitoring` — refresh statistics so planner estimates correctly
3. Partition table by month — January query only reads January partition (8M rows, not 500M)

**Result:** 45 seconds → 0.3 seconds. The same query. Same data. Same result.
Analysts could now run the report on demand. Dashboard became interactive.

**Interview Line:** *"EXPLAIN ANALYZE told me the planner estimated 50K rows but was scanning 2M.
That mismatch was the first clue — stale statistics. The second clue was Seq Scan on a 500M row table.
Two changes: composite index and ANALYZE. Query went from 45 seconds to 0.3 seconds."*

In [ ]:
# Practice: Write the SQL for these scenarios

# 1. Find the top 10 servers by average CPU over the last 7 days
# (Use window function + GROUP BY)

# 2. Find servers where CPU INCREASED by more than 20 percentage points
# between January 2024 and January 2025
# (Use self-join or LAG)

# 3. Design a composite index for this query:
# SELECT * FROM fact_monitoring
# WHERE server_id = 42 AND collected_at > NOW() - INTERVAL '30 days'
# ORDER BY collected_at DESC;
# Which columns go in the index, and in what order?

# Write your SQL here:
print("Query 1: Top 10 servers by average CPU (last 7 days)")
print("""
SELECT
    s.server_name,
    ROUND(AVG(m.metric_value), 2) AS avg_cpu
FROM fact_monitoring m
JOIN dim_server s ON m.server_id = s.server_id
WHERE m.metric_type = 'cpu_pct'
  AND m.collected_at >= NOW() - INTERVAL '7 days'
GROUP BY s.server_name
ORDER BY avg_cpu DESC
LIMIT 10;
""")

print("Index for Query 3:")
print("CREATE INDEX idx ON fact_monitoring(server_id, collected_at DESC);")
print("Reason: server_id first (equality filter), collected_at second (range + order)")

## 🎯 Summary

### The Pattern
Optimize queries by understanding the **execution plan**, not guessing.

### The Four Levers
1. **Indexes** — Direct the planner to rows without scanning the table
2. **Statistics** — `ANALYZE` keeps row count estimates accurate
3. **Partitioning** — Prune irrelevant data at the storage level
4. **Join order** — Small table as the probe side of hash joins

### When to Optimize
- Sequential scan on a table with > 100K rows in a production query
- Query planner estimates differ from actual by > 10x (stale stats)
- Query appears in the slow query log
- Report that used to run in < 1s now takes > 5s

### Interview Confidence Checklist
- [ ] Can read EXPLAIN output and identify Seq Scan vs Index Scan
- [ ] Can explain composite index column ordering rule
- [ ] Can describe Hash Join vs Nested Loop vs Merge Join
- [ ] Can name the Citi narrative: 45s → 0.3s via composite index + partition

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀